In [3]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
print('All packages loaded')

All packages loaded


# Introduction
In this notebook we calibrate a number of models on EURIBOR daily data, with the aim to compare their performance with a simple Random Walk model. The models that we are going to use can be categorized in 2 different field of study:
1. **Mathematical Finance**: Vasicek, CIR
2. **Machine Learning**: Random Forest, LSTM

For each model we perform a **rolling window** approach, with a number of **30** observations for each step, and forecast: 
*1-day, 1_week, 1_month, 3_months ahead*. 

# Import EURIBOR data
First step is importing the full dataset, clean it, and keep only the time series we are working on.

In [4]:
df = pd.read_excel('EURIBOR.xlsx')
df.head()

,Date,1 week,1 month,3 month,6 month,12 month
0,2006-04-03,2.614,2.647,2.818,2.992,3.254
1,2006-04-04,2.615,2.649,2.822,3.002,3.264
2,2006-04-05,2.635,2.651,2.824,3.006,3.260
3,2006-04-06,2.653,2.654,2.831,3.007,3.260
4,2006-04-07,2.643,2.641,2.764,2.915,3.159


Let's keep the 1 week maturity rates only, which are the best proxy for **short term rates**, and keep the Date as a singular Series. This is important as the Mathematical Finance models we use, have been theoretically built to model short term rates in continuous time. 

In [5]:
dates = df.Date
euribor_ir = df.iloc[:, 1] / 100
euribor_ir.head()

0    0.02614
1    0.02615
2    0.02635
3    0.02653
4    0.02643
Name: 1 week, dtype: float64

# Random Walk 
This simple model can be mathematically expressed as follows:

$i_{t+\delta t} = i_{t} + \epsilon$

Where $\epsilon$ is a disturbance term with zero mean. 
This allows us to employ the **Random Walk** as a straightforward benchmark, where the optimal prediction for rates at time $t+1$ is merely the rates at time $t$. In essence, our time series is inherently unpredictable, as there is no additional information that can be harnessed for forecasting purposes. 

In [6]:
df = euribor_ir.copy()
df = pd.DataFrame({'series1': df})
df = df.join(euribor_ir.shift(1),  how='left')
df.columns = ['rates t', 'rates t-1']
df = df.dropna()
df.reset_index(inplace=True)
df = df.iloc[:, 1:]
print(df.head())


# split y and X
y = df['rates t']
X = df[['rates t-1']]

   rates t  rates t-1
0  0.02615    0.02614
1  0.02635    0.02615
2  0.02653    0.02635
3  0.02643    0.02653
4  0.02630    0.02643


In [7]:
# Variables of interest for the calibration
ts_length = df.shape[0]
n_obs = 30
max_time_step = 90

pred1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred90_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact90_v = np.zeros(ts_length-(n_obs+max_time_step-1))

"""
create a for loop for the rolling window
we subtract n_obs because n is the first index of the sample, not the last one
I need to train the first 30 obs, then 2:31 etc... 
"""
for n in range(ts_length-n_obs):

    # by doing that I keep all the observations from n to (n+n_obs-1)
    X_train = X.iloc[n:n+n_obs, :]
    y_train = y.iloc[n:n+n_obs]
    prediction_index = y_train.index[-1]
    
    # Need to break the loop if my max prediction cannot be backtested
    if (prediction_index + max_time_step) > y.index[-1]:
        break
    else:
        # predictions and exact values at 4 time steps 
        r0 = y.iloc[prediction_index]

        pred1 = r0 
        pred7 = r0 
        pred30 = r0 
        pred90 = r0 

        exact1 = y.iloc[prediction_index + 1]
        exact7 = y.iloc[prediction_index + 7]
        exact30 = y.iloc[prediction_index + 30]
        exact90 = y.iloc[prediction_index + 90]

        # store the predictions and the exact values
        pred1_v[n] = pred1
        pred7_v[n] = pred7
        pred30_v[n] = pred30
        pred90_v[n] = pred90
        
        exact1_v[n] = exact1
        exact7_v[n] = exact7
        exact30_v[n] = exact30
        exact90_v[n] = exact90

In [8]:
# Create a dataframe with all the predictions and one with all exact values
pred_RW = {
    '1 step pred': pred1_v,
    '7 step pred': pred7_v,
    '30 step pred': pred30_v,
    '90 step pred': pred90_v
}

exact_rates = {
    '1 step exact': exact1_v,
    '7 step exact': exact7_v,
    '30 step exact': exact30_v,
    '90 step exact': exact90_v   
}

pred_RW = pd.DataFrame(pred_RW)
exact_rates = pd.DataFrame(exact_rates)
pred_RW.index = dates.iloc[n_obs:pred_RW.shape[0]+n_obs]
exact_rates.index = dates.iloc[n_obs:pred_RW.shape[0]+n_obs]

In [9]:
# Compute MSE
SE_RW = pd.DataFrame(columns=['1 step ahead', '7 steps ahead', '30 steps ahead', '90 steps ahead'])
for n in range(4):
    SE_RW[SE_RW.columns[n]] = (pred_RW.iloc[:, n] - exact_rates.iloc[:, n])**2

SE_RW.head()

,1 step ahead,7 steps ahead,30 steps ahead,90 steps ahead
Date,,,,
2006-05-18,0.000000e+00,9.000000e-10,0.000005,0.000019
2006-05-19,0.000000e+00,3.600000e-09,0.000005,0.000019
2006-05-22,2.500000e-09,4.000000e-10,0.000005,0.000019
2006-05-23,4.000000e-10,4.900000e-09,0.000004,0.000019
2006-05-24,1.000000e-10,1.960000e-08,0.000004,0.000021


# Vasicek Model

In [10]:
# Variables of interest for the calibration
pred1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred90_v = np.zeros(ts_length-(n_obs+max_time_step-1))

In [11]:
"""
create a for loop for the rolling window
we subtract n_obs because n is the first index of the sample, not the last one
I need to train the first 30 obs, then 2:31 etc... 
"""

for n in range(ts_length-n_obs):
    #choose the model
    model = LinearRegression() 

    # by doing that I keep all the observations from n to (n+n_obs-1)
    X_train = X.iloc[n:n+n_obs, :]
    y_train = y.iloc[n:n+n_obs]
    prediction_index = y_train.index[-1]
    
    if (prediction_index + max_time_step) > y.index[-1]:
        break
    else:
        # fit the model
        model.fit(X_train, y_train)

        # get parameters of the linear regression
        intercept = model.intercept_  
        slope = model.coef_[0]        

        # get the parameter of interest for Vasicek
        dt = 1/252                    
        k = (1-slope)/dt
        theta = intercept / (1-slope)

        # predictions at 4 time steps 
        r0 = y.iloc[prediction_index]

        pred1 = r0 * np.exp(-k*dt) + theta * (1 - np.exp(-k*dt))
        pred7 = r0 * np.exp(-k*7*dt) + theta * (1 - np.exp(-k*7*dt))
        pred30 = r0 * np.exp(-k*30*dt) + theta * (1 - np.exp(-k*30*dt))
        pred90 = r0 * np.exp(-k*90*dt) + theta * (1 - np.exp(-k*90*dt))

        # store the predictions 
        pred1_v[n] = pred1
        pred7_v[n] = pred7
        pred30_v[n] = pred30
        pred90_v[n] = pred90

In [12]:
# Create a dataframe with all the predictions and one with all exact values
pred_vsk = {
    '1 step pred': pred1_v,
    '7 step pred': pred7_v,
    '30 step pred': pred30_v,
    '90 step pred': pred90_v
}

pred_VSK = pd.DataFrame(pred_vsk)
pred_VSK.index = dates.iloc[n_obs:pred_RW.shape[0]+n_obs]

In [13]:
# Compute MSE
SE_VSK = pd.DataFrame(columns=['1 step ahead', '7 steps ahead', '30 steps ahead', '90 steps ahead'])
for n in range(4):
    SE_VSK[SE_VSK.columns[n]] = (pred_VSK.iloc[:, n] - exact_rates.iloc[:, n])**2

SE_VSK.head()

,1 step ahead,7 steps ahead,30 steps ahead,90 steps ahead
Date,,,,
2006-05-18,2.153204e-10,4.176981e-10,0.000005,0.000019
2006-05-19,2.352863e-10,6.091174e-11,0.000005,0.000018
2006-05-22,1.685673e-09,1.523556e-10,0.000005,0.000019
2006-05-23,1.586517e-10,1.971865e-09,0.000004,0.000019
2006-05-24,6.020662e-11,1.746082e-08,0.000004,0.000021


# CIR

In [14]:
# Make all the rates positive
count_negatives = 0
for value in y:
    if value < 0:
        count_negatives += 1

print(f'the number of negative values in the vector is: {count_negatives}')
shift = np.percentile(y, 99)
print(shift)

# split y and X
y_shift = y.copy() + shift
X_shift = X.copy() + shift
X_shift = X_shift['rates t-1'] # make a Series for sequent computations

count_negatives = 0
for value in y_shift:
    if value < 0:
        count_negatives += 1

print(f'the number of negative values in the vector is: {count_negatives}')

the number of negative values in the vector is: 2079
0.04398
the number of negative values in the vector is: 0


In [15]:
# Variables of interest for the calibration
pred1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred90_v = np.zeros(ts_length-(n_obs+max_time_step-1))

In [16]:
"""
create a for loop for the rolling window
we subtract n_obs because n is the first index of the sample, not the last one
I need to train the first 30 obs, then 2:31 etc... 
"""

for n in range(ts_length-n_obs):
    #choose the model
    model = LinearRegression(fit_intercept=False) 

    # by doing that I keep all the observations from n to (n+n_obs-1)
    X_train = X_shift.iloc[n:n+n_obs]
    y_train = y_shift.iloc[n:n+n_obs]
    prediction_index = y_train.index[-1]
    
    if (prediction_index + max_time_step) > y_shift.index[-1]:
        break
    else:
        y_cir = (y_train - X_train) / np.sqrt(X_train)
        z1 = dt / np.sqrt(X_train)
        z2 = dt * np.sqrt(X_train)
        X_cir = np.column_stack((z1, z2))

        model.fit(X_cir, y_cir)

        # Calculate the predicted values (y_hat), residuals and the parameters
        beta1 = model.coef_[0]        
        beta2 = model.coef_[1]

        # get the parameter of interest for CIR
        k = -beta2
        theta = beta1/k
        dt = 1/252

        # predictions at 4 time steps 
        r0 = y_shift.iloc[prediction_index]

        pred1 = r0 * np.exp(-k*dt) + theta * (1 - np.exp(-k*dt)) - shift
        pred7 = r0 * np.exp(-k*7*dt) + theta * (1 - np.exp(-k*7*dt)) - shift
        pred30 = r0 * np.exp(-k*30*dt) + theta * (1 - np.exp(-k*30*dt)) - shift
        pred90 = r0 * np.exp(-k*90*dt) + theta * (1 - np.exp(-k*90*dt)) - shift

In [17]:
# Create a dataframe with all the predictions and one with all exact values
pred_CIR = {
    '1 step pred': pred1_v,
    '7 step pred': pred7_v,
    '30 step pred': pred30_v,
    '90 step pred': pred90_v
}

pred_CIR = pd.DataFrame(pred_vsk)
pred_CIR.index = dates.iloc[n_obs:pred_RW.shape[0]+n_obs]

In [18]:
# Compute MSE
SE_CIR = pd.DataFrame(columns=['1 step ahead', '7 steps ahead', '30 steps ahead', '90 steps ahead'])
for n in range(4):
    SE_CIR[SE_CIR.columns[n]] = (pred_CIR.iloc[:, n] - exact_rates.iloc[:, n])**2

SE_CIR.head()

,1 step ahead,7 steps ahead,30 steps ahead,90 steps ahead
Date,,,,
2006-05-18,2.153204e-10,4.176981e-10,0.000005,0.000019
2006-05-19,2.352863e-10,6.091174e-11,0.000005,0.000018
2006-05-22,1.685673e-09,1.523556e-10,0.000005,0.000019
2006-05-23,1.586517e-10,1.971865e-09,0.000004,0.000019
2006-05-24,6.020662e-11,1.746082e-08,0.000004,0.000021


# Random Forest

In [ ]:
# Variables of interest for the calibration
pred1_v = np.zeros(ts_length-(n_obs+max_time_step-1))

In [ ]:
"""
create a for loop for the rolling window
we subtract n_obs because n is the first index of the sample, not the last one
I need to train the first 30 obs, then 2:31 etc... 
"""

for n in range(ts_length-n_obs):
    #choose the model
    model = RandomForestRegressor(n_estimators=25, random_state=0)
    num_lags = 5

    # by doing that I keep all the observations from n to (n+n_obs-1)
    y_train = y[n:n+n_obs]
    prediction_index = y_train.index[-1]
    
    if (prediction_index + max_time_step) > y.index[-1]:
        break
    else:
        lagged_data = pd.DataFrame()
        for lag in range(1, num_lags + 1):
            lagged_data[f"Lag_{lag}"] = y_train.shift(lag)

        # create initial df
        lagged_data = lagged_data.dropna()
        target = y_train[num_lags:]

        # reset indexes
        lagged_data.reset_index(inplace=True)
        target.reset_index(inplace=True)
        lagged_data = lagged_data.iloc[:, 1:]
        target = target.iloc[:, 1]

        # Split the data into training and testing sets (last obs is our test since working with time series)
        X_train, X_test = lagged_data.iloc[:-1, :], lagged_data.iloc[-1, :]
        y_train, y_test = target.iloc[:-1], target.iloc[-1]

        # Train a random forest model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        pred1 = model.predict(X_test.to_frame().T)

        # store the predictions 
        pred1_v[n] = pred1

In [ ]:
# Create a dataframe with all the predictions and one with all exact values
pred_rf = {
    '1 step pred': pred1_v,
}

pred_rf = pd.DataFrame(pred_rf)
pred_rf.index = dates.iloc[n_obs:pred_RW.shape[0]+n_obs]

In [ ]:
# Compute MSE
SE_rf = pd.DataFrame(columns=['1 step ahead'])
SE_rf['1 step ahead'] = (pred_rf.iloc[:, 0] - exact_rates.iloc[:, 0])**2

SE_rf.head()

# LSTM